In [33]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score, classification_report
from ast import literal_eval
import joblib
import os

# создаём папку, если её нет
os.makedirs("xgb_full", exist_ok=True)

# ==============================
# 1. Загружаем данные
# ==============================
data = pd.read_csv('train_ensamble.csv')
for col in data.columns:
    if col == 'sample': 
        continue
    data[col] = data[col].apply(lambda x: literal_eval(x))

X_list, y_list = [], []

# ==============================
# 2. Формируем фичи и метки
# ==============================
for _, row in data.iterrows():
    words = row['sample'].split()
    n_words = len(words)

    for i, word in enumerate(words):
        # текущие вероятности
        probs = [
            row['brand_proba_1'][i],
            row['brand_proba_2'][i],
            row['type_proba_1'][i],
            row['type_proba_2'][i],
            row['percent_proba_1'][i],
            row['percent_proba_2'][i],
            row['volume_proba_1'][i],
            row['volume_proba_2'][i],
            row['o_probs'][i],
        ]

        # признаки слова
        word_pos = i
        word_len = len(word)

        # сосед слева
        if i > 0:
            prev_probs = [
                row['brand_proba_1'][i-1],
                row['brand_proba_2'][i-1],
                row['type_proba_1'][i-1],
                row['type_proba_2'][i-1],
                row['percent_proba_1'][i-1],
                row['percent_proba_2'][i-1],
                row['volume_proba_1'][i-1],
                row['volume_proba_2'][i-1],
                row['o_probs'][i-1],
            ]
            has_prev = 1
        else:
            prev_probs = [0] * 9
            has_prev = 0

        # сосед справа
        if i < n_words - 1:
            next_probs = [
                row['brand_proba_1'][i+1],
                row['brand_proba_2'][i+1],
                row['type_proba_1'][i+1],
                row['type_proba_2'][i+1],
                row['percent_proba_1'][i+1],
                row['percent_proba_2'][i+1],
                row['volume_proba_1'][i+1],
                row['volume_proba_2'][i+1],
                row['o_probs'][i+1],
            ]
            has_next = 1
        else:
            next_probs = [0] * 9
            has_next = 0

        # объединяем фичи
        features = probs + [word_pos, word_len] + prev_probs + [has_prev] + next_probs + [has_next]
        X_list.append(features)

    # метки
    labels = []
    for i, word in enumerate(words):
        label = 'O'
        word_start = sum(len(w)+1 for w in words[:i])
        word_end = word_start + len(word)

        for start, end, tag in row['annotation']:
            if start <= word_start < end:
                label = tag

        # for start, end, tag in row['annotation']:
        #     if start <= word_start < end:
        #         if tag.endswith('B-BRAND'):
        #             label = 'B-BRAND'
        #         elif tag.endswith('TYPE'):
        #             label = 'TYPE'
        #         elif tag.endswith('PERCENT'):
        #             label = 'PERCENT'
        #         elif tag.endswith('VOLUME'):
        #             label = 'VOLUME'
        labels.append(label)

    y_list.extend(labels)

# ==============================
# 3. NumPy массивы
# ==============================
X = np.array(X_list, dtype=np.float32)
y = np.array(y_list)

# ==============================
# 4. LabelEncoder
# ==============================
le = LabelEncoder()
y_enc = le.fit_transform(y)

print("Classes in LabelEncoder:", le.classes_)
print("Unique labels after transform:", np.unique(y_enc))

# ==============================
# 5. Веса классов
# ==============================
classes, counts = np.unique(y_enc, return_counts=True)
class_weights = {cls: 1.0 / cnt for cls, cnt in zip(classes, counts)}
mean_w = np.mean(list(class_weights.values()))
class_weights = {cls: w / mean_w for cls, w in class_weights.items()}
sample_weight = np.array([class_weights[label] for label in y_enc])

# ==============================
# 6. KFold обучение без валидации
# ==============================
kf = KFold(n_splits=4, shuffle=True, random_state=42)
macro_f1_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    print(f"\n===== FOLD {fold} =====")

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y_enc[train_idx], y_enc[val_idx]
    w_train = sample_weight[train_idx]

    clf = XGBClassifier(
        objective="multi:softprob",
        num_class=len(np.unique(y_enc)),
        eval_metric="mlogloss",
        n_estimators=300,
        learning_rate=0.1,
        max_depth=3,
        random_state=42,
        n_jobs=-1,
    )

    clf.fit(X_train, y_train, sample_weight=w_train, eval_set=[(X_val, y_val)], verbose=False)

    y_pred = clf.predict(X_val)

    print(f"\nFold {fold} classification report:\n")
    print(classification_report(
        y_val, 
        y_pred, 
        labels=np.arange(len(le.classes_)),  # всегда [0..8]
        target_names=le.classes_
    ))

    macro_f1 = f1_score(y_val, y_pred, average='macro')
    print(f"Fold {fold} Macro F1-score: {macro_f1:.4f}")
    macro_f1_scores.append(macro_f1)

    joblib.dump(clf, f"xgb_full/xgb_fold{fold}.joblib")
    print(f"Fold {fold} model saved to xgb_full/xgb_fold{fold}.joblib")

# ==============================
# 7. Итог
# ==============================
print(f"\nAverage Macro F1 over 4 folds: {np.mean(macro_f1_scores):.4f}")
joblib.dump(le, "xgb_full/label_encoder.joblib")
print("LabelEncoder saved.")


Classes in LabelEncoder: ['B-BRAND' 'B-PERCENT' 'B-TYPE' 'B-VOLUME' 'I-BRAND' 'I-PERCENT' 'I-TYPE'
 'I-VOLUME' 'O']
Unique labels after transform: [0 1 2 3 4 5 6 7 8]

===== FOLD 1 =====

Fold 1 classification report:

              precision    recall  f1-score   support

     B-BRAND       0.97      0.99      0.98      1828
   B-PERCENT       0.83      0.83      0.83         6
      B-TYPE       1.00      0.99      0.99      6135
    B-VOLUME       0.90      1.00      0.95        19
     I-BRAND       0.91      0.99      0.95       130
   I-PERCENT       0.00      0.00      0.00         0
      I-TYPE       0.98      0.98      0.98      1115
    I-VOLUME       1.00      1.00      1.00         7
           O       0.98      0.98      0.98      1337

    accuracy                           0.99     10577
   macro avg       0.84      0.86      0.85     10577
weighted avg       0.99      0.99      0.99     10577

Fold 1 Macro F1-score: 0.9580
Fold 1 model saved to xgb_full/xgb_fold1.jobli

/home/dmitry/venvs/GNN/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/dmitry/venvs/GNN/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/dmitry/venvs/GNN/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/h


Fold 2 classification report:

              precision    recall  f1-score   support

     B-BRAND       0.98      0.98      0.98      1819
   B-PERCENT       0.71      1.00      0.83         5
      B-TYPE       1.00      0.99      0.99      6158
    B-VOLUME       0.80      0.89      0.84         9
     I-BRAND       0.86      1.00      0.92       126
   I-PERCENT       1.00      1.00      1.00         2
      I-TYPE       0.98      0.98      0.98      1089
    I-VOLUME       1.00      1.00      1.00         5
           O       0.98      0.99      0.98      1363

    accuracy                           0.99     10576
   macro avg       0.92      0.98      0.95     10576
weighted avg       0.99      0.99      0.99     10576

Fold 2 Macro F1-score: 0.9486
Fold 2 model saved to xgb_full/xgb_fold2.joblib

===== FOLD 3 =====

Fold 3 classification report:

              precision    recall  f1-score   support

     B-BRAND       0.97      0.99      0.98      1806
   B-PERCENT       0.88 

/home/dmitry/venvs/GNN/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/dmitry/venvs/GNN/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/dmitry/venvs/GNN/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



Fold 4 classification report:

              precision    recall  f1-score   support

     B-BRAND       0.98      0.98      0.98      1835
   B-PERCENT       0.56      1.00      0.71         5
      B-TYPE       1.00      0.99      0.99      6085
    B-VOLUME       0.92      0.75      0.83        16
     I-BRAND       0.88      0.99      0.93       129
   I-PERCENT       1.00      1.00      1.00         2
      I-TYPE       0.97      0.99      0.98      1160
    I-VOLUME       1.00      1.00      1.00         8
           O       0.98      0.98      0.98      1336

    accuracy                           0.99     10576
   macro avg       0.92      0.97      0.93     10576
weighted avg       0.99      0.99      0.99     10576

Fold 4 Macro F1-score: 0.9347
Fold 4 model saved to xgb_full/xgb_fold4.joblib

Average Macro F1 over 5 folds: 0.9179
LabelEncoder saved.


In [35]:
import numpy as np
import pandas as pd
import joblib
from ast import literal_eval
import re
import torch

# ======= Rule-based prior =======
VOLUME_RE  = re.compile(r"(?<!\w)(\d+[.,]?\d*)\s?(л|л\.|литр\w*|мл|ml|г|кг|шт|уп\w*|пак\w*|бут\w*)(?!\w)", re.IGNORECASE)
PERCENT_RE = re.compile(r"(?<!\w)(\d+[.,]?\d*)\s?%|процент\w*", re.IGNORECASE)

def rule_spans(text):
    out=[]
    for m in VOLUME_RE.finditer(text): out.append((m.start(), m.end(), "VOLUME"))
    for m in PERCENT_RE.finditer(text): out.append((m.start(), m.end(), "PERCENT"))
    return out

def apply_priors_to_probs(text, words, probs_row, beta=2.0):
    """
    words: list of слов
    probs_row: np.array, shape = (n_words, n_labels) с вероятностями моделей
    """
    offsets = []
    char_idx = 0
    for w in words:
        start = text.find(w, char_idx)
        end = start + len(w)
        offsets.append((start, end))
        char_idx = end + 1

    spans = rule_spans(text)
    for s, e, t in spans:
        for i, (w_start, w_end) in enumerate(offsets):
            if max(s, w_start) < min(e, w_end):  # пересечение
                if t == "VOLUME":
                    probs_row[i][3] += beta
                    probs_row[i][7] += beta
                elif t == "PERCENT":
                    probs_row[i][1] += beta
                    probs_row[i][5] += beta
    return probs_row

# --------------------------
# 1. Загружаем тестовый датасет
# --------------------------
test_data = pd.read_csv('test_ensamble.csv')
for col in test_data.columns:
    if col == 'sample':
        continue
    test_data[col] = test_data[col].apply(lambda x: literal_eval(x))

# --------------------------
# 2. Загружаем модели и LabelEncoder
# --------------------------
models = [joblib.load(f"xgb_full/xgb_fold{fold}.joblib") for fold in range(1, 6)]
le = joblib.load("xgb_full/label_encoder.joblib")

# --------------------------
# 3. Функция инференса с ансамблем
# --------------------------
def infer_annotations_ensemble(models, df, label_encoder):
    all_annotations = []

    for idx, row in df.iterrows():
        text = row['sample']
        words = text.split()
        n_words = len(words)

        # --------------------------
        # собираем фичи для каждого слова
        # --------------------------
        features = []
        for i, word in enumerate(words):
            # текущие вероятности
            probs = [
                row['brand_proba_1'][i],
                row['brand_proba_2'][i],
                row['type_proba_1'][i],
                row['type_proba_2'][i],
                row['percent_proba_1'][i],
                row['percent_proba_2'][i],
                row['volume_proba_1'][i],
                row['volume_proba_2'][i],
                row['o_probs'][i],
            ]
    
            # признаки слова
            word_pos = i
            word_len = len(word)
    
            # сосед слева
            if i > 0:
                prev_probs = [
                    row['brand_proba_1'][i-1],
                    row['brand_proba_2'][i-1],
                    row['type_proba_1'][i-1],
                    row['type_proba_2'][i-1],
                    row['percent_proba_1'][i-1],
                    row['percent_proba_2'][i-1],
                    row['volume_proba_1'][i-1],
                    row['volume_proba_2'][i-1],
                    row['o_probs'][i-1],
                ]
                has_prev = 1
            else:
                prev_probs = [0] * 9
                has_prev = 0
    
            # сосед справа
            if i < n_words - 1:
                next_probs = [
                    row['brand_proba_1'][i+1],
                    row['brand_proba_2'][i+1],
                    row['type_proba_1'][i+1],
                    row['type_proba_2'][i+1],
                    row['percent_proba_1'][i+1],
                    row['percent_proba_2'][i+1],
                    row['volume_proba_1'][i+1],
                    row['volume_proba_2'][i+1],
                    row['o_probs'][i+1],
                ]
                has_next = 1
            else:
                next_probs = [0] * 9
                has_next = 0


            feat = probs + [word_pos, word_len] + prev_probs + [has_prev] + next_probs + [has_next]
            features.append(feat)

        X_test = np.array(features, dtype=np.float32)

        # --------------------------
        # 3a. Получаем прогноз каждой модели
        # --------------------------
        all_probs = []
        for model in models:
            probs = model.predict_proba(X_test)  # shape = (n_words, n_classes)
            all_probs.append(probs)
        
        # --------------------------
        # 3b. Усредняем вероятности (soft voting)
        # --------------------------
        mean_probs = np.mean(all_probs, axis=0)
        mean_probs = apply_priors_to_probs(text, words, mean_probs, beta=1.0)
        
        # предсказанные индексы и метки
        y_pred_num = mean_probs.argmax(axis=1)
        y_pred_labels = label_encoder.inverse_transform(y_pred_num)  # уже BIO: B-BRAND, I-TYPE, O и т.д.
        
        # --------------------------
        # 3c. Формируем аннотации
        # --------------------------
        annotations = []
        char_idx = 0
        
        for word, label in zip(words, y_pred_labels):
            start_idx = text.find(word, char_idx)
            end_idx = start_idx + len(word)
            char_idx = end_idx + 1  # сдвигаем для поиска следующего слова
        
            annotations.append((start_idx, end_idx, label))
        
        all_annotations.append(annotations)

    df['annotation'] = all_annotations
    return df

# --------------------------
# 4. Запускаем инференс
# --------------------------
test_data = infer_annotations_ensemble(models, test_data, le)

# --------------------------
# 5. Сохраняем результат
# --------------------------
test_data[['sample', 'annotation']].to_csv("xgboost_test_ensemble_full.csv", sep=";", index=False)
print("Inference complete. Saved to xgboost_test_ensemble.csv")


Inference complete. Saved to xgboost_test_ensemble.csv


In [8]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from ast import literal_eval
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import KFold
import joblib
import os

# создаём папку, если её нет
os.makedirs("xgb_jumbo", exist_ok=True)

data = pd.read_csv('all_labels_train_jumbo.csv')
for col in data.columns:
    if col == 'sample': continue
    data[col] = data[col].apply(lambda x: literal_eval(x))

# --------------------------
# 1. Подготовка данных
# --------------------------
X_list = []
y_list = []

for _, row in data.iterrows():
    words = row['sample'].split()
    n_words = len(words)
    
    # собираем вероятности каждого слова
    for i in range(n_words):
        probs = [
            row['brand_probs'][i],
            row['type_probs'][i],
            row['percent_probs'][i],
            row['volume_probs'][i],
            row['o_probs'][i],
            row['brand_probs_single'][i],
            row['types_probs_single'][i],
            row['o_probs_single'][i]
        ]
        X_list.append(probs)
        
    # собираем метки слов, объединяя B/I в один тип
    labels = []
    for i, word in enumerate(words):
        label = 'O'
        for start, end, tag in row['annotation']:
            if start <= sum(len(w)+1 for w in words[:i]) < end:  # если слово попадает в диапазон
                if tag.endswith('BRAND'):
                    label = 'BRAND'
                elif tag.endswith('TYPE'):
                    label = 'TYPE'
                elif tag.endswith('PERCENT'):
                    label = 'PERCENT'
                elif tag.endswith('VOLUME'):
                    label = 'VOLUME'
        labels.append(label)
    y_list.extend(labels)

# --------------------------
# 1. Преобразуем X и y
# --------------------------
X = np.array(X_list, dtype=np.float32)
y = np.array(y_list)

# --------------------------
# 2. Кодирование меток
# --------------------------
le = LabelEncoder()
y_enc = le.fit_transform(y)

# ==============================
# 3. Считаем веса классов
# ==============================
classes, counts = np.unique(y_enc, return_counts=True)
class_weights = {cls: 1.0 / cnt for cls, cnt in zip(classes, counts)}

# нормализуем, чтобы средний вес был = 1
mean_w = np.mean(list(class_weights.values()))
class_weights = {cls: w / mean_w for cls, w in class_weights.items()}

# переводим y -> sample_weight
sample_weight = np.array([class_weights[label] for label in y_enc])

# --------------------------
# 3. KFold 5
# --------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
macro_f1_scores = []
models = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"\n===== FOLD {fold+1} =====")

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y_enc[train_idx], y_enc[val_idx]
    w_train = sample_weight[train_idx]

    clf = XGBClassifier(
        objective="multi:softmax",
        num_class=len(le.classes_),
        eval_metric="mlogloss",
        n_estimators=400,
        learning_rate=0.1,
        max_depth=4,
        random_state=42,
        n_jobs=-1,
    )

    clf.fit(X_train, y_train, sample_weight=w_train, 
            eval_set=[(X_val, y_val)], verbose=False)

    # предсказания
    y_pred = clf.predict(X_val)

    # --------------------------
    # 6. F1-score
    # --------------------------
    print(f"\nFold {fold} classification report:\n")
    print(classification_report(y_val, y_pred, target_names=le.classes_))

    macro_f1 = f1_score(y_val, y_pred, average='macro')
    print(f"Fold {fold} Macro F1-score: {macro_f1:.4f}")
    macro_f1_scores.append(macro_f1)
    fold += 1

    # --------------------------
    # 7. Сохраняем модель
    # --------------------------
    model_path = f"xgb_jumbo/xgb_fold{fold}.joblib"
    joblib.dump(clf, model_path)
    print(f"Fold {fold} model saved to {model_path}")

    models.append(clf)

# --------------------------
# 7. Среднее макро-F1 по всем фолдам
# --------------------------
print(f"\nAverage Macro F1 over 5 folds: {np.mean(macro_f1_scores):.4f}")
joblib.dump(le, "xgb_jumbo/label_encoder.joblib")
print("LabelEncoder saved.")


===== FOLD 1 =====

Fold 0 classification report:

              precision    recall  f1-score   support

       BRAND       0.98      0.99      0.99      1547
           O       0.96      0.98      0.97      1071
     PERCENT       1.00      1.00      1.00         8
        TYPE       1.00      0.99      0.99      5806
      VOLUME       0.85      0.89      0.87        19

    accuracy                           0.99      8451
   macro avg       0.96      0.97      0.96      8451
weighted avg       0.99      0.99      0.99      8451

Fold 0 Macro F1-score: 0.9638
Fold 1 model saved to xgb_jumbo/xgb_fold1.joblib

===== FOLD 2 =====

Fold 1 classification report:

              precision    recall  f1-score   support

       BRAND       0.98      0.99      0.99      1501
           O       0.95      0.99      0.97      1067
     PERCENT       0.57      1.00      0.73         4
        TYPE       1.00      0.99      0.99      5865
      VOLUME       1.00      0.93      0.96        14

  

In [10]:
import re
import numpy as np
import pandas as pd
import joblib
from ast import literal_eval

# --------------------------
# 1. Загружаем тестовый датасет
# --------------------------
test_data = pd.read_csv('all_labels_test_jumbo.csv')
for col in test_data.columns:
    if col == 'sample':
        continue
    test_data[col] = test_data[col].apply(lambda x: literal_eval(x))

# --------------------------
# 2. Загружаем модели и LabelEncoder
# --------------------------
models = [joblib.load(f"xgb_jumbo/xgb_fold{fold}.joblib") for fold in range(1, 6)]
le = joblib.load("xgb_jumbo/label_encoder.joblib")

# --------------------------
# 3. Функция инференса с ансамблем
# --------------------------
def infer_annotations_ensemble(models, df, label_encoder):
    all_annotations = []

    for idx, row in df.iterrows():
        text = row['sample']
        words = text.split()
        n_words = len(words)

        # собираем фичи для каждого слова
        X_test = np.array([
            [
            row['brand_probs'][i],
            row['type_probs'][i],
            row['percent_probs'][i],
            row['volume_probs'][i],
            row['o_probs'][i],
            row['brand_probs_single'][i],
            row['types_probs_single'][i],
            row['o_probs_single'][i]
            ] for i in range(n_words)
        ], dtype=np.float32)

        # --------------------------
        # 3a. Получаем прогноз каждой модели
        # --------------------------
        all_probs = []
        for model in models:
            probs = model.predict_proba(X_test)  # shape = (n_words, n_classes)
            all_probs.append(probs)

        # --------------------------
        # 3b. Усредняем вероятности (soft voting)
        # --------------------------
        mean_probs = np.mean(all_probs, axis=0)
        y_pred_num = mean_probs.argmax(axis=1)
        y_pred_labels = label_encoder.inverse_transform(y_pred_num)

        # --------------------------
        # 3c. Формируем BIO-аннотации
        # --------------------------
        annotations = []
        prev_label = None
        char_idx = 0

        for i, word in enumerate(words):
            start_idx = text.find(word, char_idx)
            end_idx = start_idx + len(word)
            char_idx = end_idx + 1  # сдвигаем на 1 для пробела

            label = y_pred_labels[i]
            if label == 'O':
                bio_label = 'O'
                prev_label = None
            elif prev_label == label:
                bio_label = f'I-{label}'
            else:
                bio_label = f'B-{label}'

            prev_label = label
            annotations.append((start_idx, end_idx, bio_label))

        all_annotations.append(annotations)

    df['annotation'] = all_annotations
    return df

# --------------------------
# 4. Запускаем инференс
# --------------------------
test_data = infer_annotations_ensemble(models, test_data, le)

# --------------------------
# 5. Сохраняем результат
# --------------------------
test_data[['sample', 'annotation']].to_csv("xgboost_test_ensemble_jumbo.csv", sep=";", index=False)
print("Inference complete. Saved to xgboost_test_ensemble_jumbo.csv")


Inference complete. Saved to xgboost_test_ensemble_jumbo.csv


In [67]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn_crfsuite import CRF
from sklearn_crfsuite.metrics import flat_classification_report


# --------------------------
# 1. Загружаем датасет
# --------------------------
df = pd.read_csv("all_labels_train_jumbo.csv")
for col in df.columns:
    if col == 'sample': continue
    df[col] = df[col].apply(lambda x: literal_eval(x))

# ===============================
# 2. Функции для обработки
# ===============================
def extract_features(words, i):
    """Признаки для CRF"""
    word = words[i]
    features = {
        'word.lower()': word.lower(),
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
    }
    if i > 0:
        word1 = words[i-1]
        features.update({
            '-1:word.lower()': word1.lower(),
            '-1:word.istitle()': word1.istitle(),
            '-1:word.isupper()': word1.isupper(),
        })
    else:
        features['BOS'] = True  # начало строки

    if i < len(words)-1:
        word1 = words[i+1]
        features.update({
            '+1:word.lower()': word1.lower(),
            '+1:word.istitle()': word1.istitle(),
            '+1:word.isupper()': word1.isupper(),
        })
    else:
        features['EOS'] = True  # конец строки
    return features

def sent2features(words):
    return [extract_features(words, i) for i in range(len(words))]

def sent2labels(row):
    words = row['sample'].split()
    labels = []
    char_pos_list = [sum(len(w) + 1 for w in words[:i]) for i in range(len(words))]

    for i, char_pos in enumerate(char_pos_list):
        label = 'O'
        for start, end, tag in row['annotation']:  # <--- без eval
            if start <= char_pos < end:  # если слово попадает в диапазон
                if tag.endswith('BRAND'):
                    label = 'BRAND'
                elif tag.endswith('TYPE'):
                    label = 'TYPE'
                elif tag.endswith('PERCENT'):
                    label = 'PERCENT'
                elif tag.endswith('VOLUME'):
                    label = 'VOLUME'
        labels.append(label)
    return labels

# ===============================
# 3. Формируем датасет
# ===============================
X = []
y = []

for _, row in df.iterrows():
    words = row['sample'].split()
    X.append(sent2features(words))
    y.append(sent2labels(row))

# ===============================
# 4. Трейн/тест
# ===============================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ===============================
# 5. Обучение CRF
# ===============================
crf = CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=1300,
    all_possible_transitions=True
)
crf.fit(X_train, y_train)

# ===============================
# 6. Предсказания
# ===============================
y_pred = crf.predict(X_test)

# ===============================
# 7. Метрики (F1 по каждому лейблу и macro)
# ===============================
labels = ['BRAND', 'TYPE', 'PERCENT', 'VOLUME', 'O']
print(flat_classification_report(y_test, y_pred, labels=labels, digits=3))
# --------------------------
# 8. Сохраняем модель
# --------------------------
# joblib.dump(crf, "crf_model.joblib")
# print("CRF model saved to crf_model.joblib")


              precision    recall  f1-score   support

       BRAND      0.926     0.631     0.751      1532
        TYPE      0.877     0.982     0.927      5816
     PERCENT      1.000     1.000     1.000         1
      VOLUME      0.842     0.516     0.640        31
           O      0.888     0.733     0.803      1103

    accuracy                          0.884      8483
   macro avg      0.907     0.773     0.824      8483
weighted avg      0.887     0.884     0.878      8483



In [68]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score, classification_report
from ast import literal_eval
import joblib
import os

# --------------------------
# Создаём папку
# --------------------------
os.makedirs("catboost", exist_ok=True)

# --------------------------
# Загружаем данные
# --------------------------
data = pd.read_csv('all_labels_train.csv')
for col in data.columns:
    if col == 'sample':
        continue
    data[col] = data[col].apply(lambda x: literal_eval(x))

# --------------------------
# 1. Подготовка данных
# --------------------------
X_list = []
y_list = []

for _, row in data.iterrows():
    words = row['sample'].split()
    n_words = len(words)

    for i, word in enumerate(words):
        # текущие фичи
        probs = [
            row['brand_probs'][i],
            row['type_probs'][i],
            row['percent_probs'][i],
            row['volume_probs'][i],
            row['o_probs'][i],
        ]
        word_pos = [i, len(word)]

        # предыдущие
        if i > 0:
            prev = [
                row['brand_probs'][i-1],
                row['type_probs'][i-1],
                row['percent_probs'][i-1],
                row['volume_probs'][i-1],
                row['o_probs'][i-1],
                i-1, len(words[i-1]),
                1  # has_prev
            ]
        else:
            prev = [0]*7 + [0]

        # следующие
        if i < n_words-1:
            nxt = [
                row['brand_probs'][i+1],
                row['type_probs'][i+1],
                row['percent_probs'][i+1],
                row['volume_probs'][i+1],
                row['o_probs'][i+1],
                i+1, len(words[i+1]),
                1  # has_next
            ]
        else:
            nxt = [0]*7 + [0]

        X_list.append(probs + word_pos + prev + nxt)

    # метки слов (BIO → тип сущности)
    labels = []
    for i, word in enumerate(words):
        label = 'O'
        for start, end, tag in row['annotation']:
            if start <= sum(len(w)+1 for w in words[:i]) < end:
                if tag.endswith('BRAND'):
                    label = 'BRAND'
                elif tag.endswith('TYPE'):
                    label = 'TYPE'
                elif tag.endswith('PERCENT'):
                    label = 'PERCENT'
                elif tag.endswith('VOLUME'):
                    label = 'VOLUME'
        labels.append(label)
    y_list.extend(labels)

# --------------------------
# 2. Преобразуем X и y
# --------------------------
X = np.array(X_list, dtype=np.float32)
y = np.array(y_list)

# --------------------------
# 3. Кодирование меток
# --------------------------
le = LabelEncoder()
y_enc = le.fit_transform(y)

# ==============================
# 4. Считаем веса классов
# ==============================
classes, counts = np.unique(y_enc, return_counts=True)
class_weights = {cls: 1.0 / cnt for cls, cnt in zip(classes, counts)}

mean_w = np.mean(list(class_weights.values()))
class_weights = {cls: w / mean_w for cls, w in class_weights.items()}

sample_weight = np.array([class_weights[label] for label in y_enc])

# --------------------------
# 5. KFold 5
# --------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)
macro_f1_scores = []
models = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    print(f"\n===== FOLD {fold} =====")

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y_enc[train_idx], y_enc[val_idx]
    w_train = sample_weight[train_idx]

    clf = CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="TotalF1",
        iterations=1500,
        learning_rate=0.1,
        depth=8,
        random_seed=42,
        verbose=False,
#        class_weights=[class_weights.get(i, 1.0) for i in range(len(le.classes_))]
    )

    clf.fit(X_train, y_train, eval_set=(X_val, y_val))

    # предсказания
    y_pred = clf.predict(X_val).astype(int).flatten()

    print(f"\nFold {fold} classification report:\n")
    print(classification_report(y_val, y_pred, target_names=le.classes_))

    macro_f1 = f1_score(y_val, y_pred, average='macro')
    print(f"Fold {fold} Macro F1-score: {macro_f1:.4f}")
    macro_f1_scores.append(macro_f1)

    # сохраняем модель
    model_path = f"catboost/cat_fold{fold}.cbm"
    clf.save_model(model_path)
    print(f"Fold {fold} model saved to {model_path}")

    models.append(clf)

# --------------------------
# 6. Среднее макро-F1
# --------------------------
print(f"\nAverage Macro F1 over 5 folds: {np.mean(macro_f1_scores):.4f}")
joblib.dump(le, "catboost/label_encoder.joblib")
print("LabelEncoder saved.")



===== FOLD 1 =====


CatBoostError: tools/enum_parser/enum_serialization_runtime/enum_runtime.cpp:70: Key 'MacroF1' not found in enum ELossFunction. Valid options are: 'Logloss', 'CrossEntropy', 'CtrFactor', 'Focal', 'RMSE', 'LogCosh', 'Lq', 'MAE', 'Quantile', 'MultiQuantile', 'Expectile', 'LogLinQuantile', 'MAPE', 'Poisson', 'MSLE', 'MedianAbsoluteError', 'SMAPE', 'Huber', 'Tweedie', 'Cox', 'RMSEWithUncertainty', 'MultiClass', 'MultiClassOneVsAll', 'PairLogit', 'PairLogitPairwise', 'YetiRank', 'YetiRankPairwise', 'QueryRMSE', 'GroupQuantile', 'QuerySoftMax', 'QueryCrossEntropy', 'StochasticFilter', 'LambdaMart', 'StochasticRank', 'PythonUserDefinedPerObject', 'PythonUserDefinedMultiTarget', 'UserPerObjMetric', 'UserQuerywiseMetric', 'R2', 'NumErrors', 'FairLoss', 'AUC', 'Accuracy', 'BalancedAccuracy', 'BalancedErrorRate', 'BrierScore', 'Precision', 'Recall', 'F1', 'TotalF1', 'F', 'MCC', 'ZeroOneLoss', 'HammingLoss', 'HingeLoss', 'Kappa', 'WKappa', 'LogLikelihoodOfPrediction', 'NormalizedGini', 'PRAUC', 'P

In [19]:
test_data.shape

(5000, 11)